# Trie (Prefix Tree)

## What is a Trie?

A **Trie** (pronounced "try") is a tree-like data structure used for efficient storage and retrieval of strings. The name comes from "re**TRIE**val" - it was designed for fast information retrieval.

### Key Properties:
- Each node represents a **single character**
- The **path** from root to any node represents a **prefix**
- Words are marked with a special **end-of-word flag** (`is_end`)
- All descendants of a node share a common prefix

### Why Use a Trie?
- **Fast prefix operations**: O(m) where m is the word/prefix length
- **Efficient autocomplete**: Find all words with a given prefix
- **Memory efficient for shared prefixes**: "cat", "car", "card" share "ca"

---[< Previous: DFA-Based Pattern Matching](../07_String_Algorithms/04_dfa_matching.ipynb) | [Tier 4: Algorithms & Data Structures Overview](../README.md) | [Next: Aho-Corasick Algorithm >](02_aho_corasick.ipynb)---

---

## Trie Structure Visualization

**Trie containing words: "cat", "car", "card", "dog", "do"**

```
           (root)
          /      \
         c        d
         |        |
         a        o
        / \       |*
       t*  r*     g*
           |
           d*

* = end of word (is_end = True)

Path "c-a-t" = "cat"
Path "c-a-r" = "car"
Path "c-a-r-d" = "card"
Path "d-o" = "do"
Path "d-o-g" = "dog"
```

Notice how:
- "cat" and "car" share the prefix "ca"
- "car" and "card" share the prefix "car"
- "do" and "dog" share the prefix "do"

---

## Trie Operations

### 1. Insert Operation

**Insert "cats" into the trie:**

```
Before:              After:
    (root)              (root)
       |                   |
       c                   c
       |                   |
       a                   a
      / \                 / \
     t*  r*              t*  r*
                         |
                         s*
```

**Algorithm:**
1. Start at root
2. For each character in word:
   - If child exists → move to it
   - Else → create new node, then move to it
3. Mark final node as end-of-word

---

### 2. Search Operation

**Search for "car":**

```
    (root)
       ↓ 'c'
       c
       ↓ 'a'
       a
      / ↓ 'r'
     t   r* ← is_end=True → FOUND!
```

**Algorithm:**
1. Start at root
2. For each character:
   - If child doesn't exist → return False
   - Else → move to child
3. Return True only if `is_end` is True

---

### 3. Prefix Search

**Find all words with prefix "ca":**

```
    (root)
       ↓
       c
       ↓
       a  ← reached prefix
      / \
     t*  r*  ← collect all words from here
         |
         d*

Result: ["cat", "car", "card"]
```

**Algorithm:**
1. Navigate to prefix end node
2. DFS/BFS to collect all words with `is_end=True`

---

### 4. Delete Operation

**Delete "car" (but keep "card"):**

```
Before:              After:
       a                   a
      / \                 / \
     t*  r*              t*  r   ← is_end=False now
         |                   |
         d*                  d*
```

Just unmark `is_end` - don't remove nodes that have children!

---

## Complexity Analysis

| Operation | Time Complexity | Space Complexity |
|-----------|-----------------|------------------|
| Insert | O(m) | O(m) worst case |
| Search | O(m) | O(1) |
| Prefix Search | O(p + k) | O(k) for results |
| Delete | O(m) | O(1) |
| Build from n words | O(n × m) | O(n × m) worst |

Where:
- **m** = length of the word
- **p** = length of the prefix
- **k** = number of matching results

### Comparison with Hash Table

| Aspect | Trie | Hash Table |
|--------|------|------------|
| Search | O(m) | O(m) average |
| Prefix search | O(p + k) | O(n × m) |
| Sorted iteration | Natural | Requires sort |
| Memory | Can be higher | Usually lower |

---

## Implementation

In [ ]:
class TrieNode:
    """
    A node in the Trie structure.
    
    Attributes:
        children (dict): Maps characters to child TrieNode objects
        is_end (bool): True if this node marks the end of a valid word
    
    Based on Node class pattern from Algorithms_HW9.ipynb
    """
    
    def __init__(self):
        self.children = {}  # char -> TrieNode
        self.is_end = False  # marks end of word
    
    def __repr__(self):
        return f"TrieNode(children={list(self.children.keys())}, is_end={self.is_end})"

In [ ]:
class Trie:
    """
    A Trie (prefix tree) data structure for efficient string operations.
    
    Supports:
        - Insert words
        - Search for exact words
        - Check if prefix exists
        - Get all words with a given prefix
        - Delete words
        - Count words with prefix
    
    Example:
        >>> trie = Trie()
        >>> trie.insert("hello")
        >>> trie.search("hello")
        True
        >>> trie.starts_with("hel")
        True
    """
    
    def __init__(self):
        """Initialize an empty Trie with a root node."""
        self.root = TrieNode()
    
    def insert(self, word: str) -> None:
        """
        Insert a word into the trie.
        
        Args:
            word: The string to insert
            
        Time Complexity: O(m) where m is word length
        Space Complexity: O(m) worst case (no shared prefix)
        """
        node = self.root
        for char in word:
            if char not in node.children:
                node.children[char] = TrieNode()
            node = node.children[char]
        node.is_end = True
    
    def search(self, word: str) -> bool:
        """
        Check if word exists in the trie.
        
        Args:
            word: The string to search for
            
        Returns:
            True if the exact word exists, False otherwise
            
        Time Complexity: O(m)
        """
        node = self._find_node(word)
        return node is not None and node.is_end
    
    def starts_with(self, prefix: str) -> bool:
        """
        Check if any word in the trie starts with the given prefix.
        
        Args:
            prefix: The prefix string to check
            
        Returns:
            True if any word starts with prefix, False otherwise
            
        Time Complexity: O(p) where p is prefix length
        """
        return self._find_node(prefix) is not None
    
    def _find_node(self, prefix: str) -> TrieNode:
        """
        Navigate to the node representing the end of prefix.
        
        Args:
            prefix: The prefix to navigate to
            
        Returns:
            The TrieNode at end of prefix, or None if prefix doesn't exist
        """
        node = self.root
        for char in prefix:
            if char not in node.children:
                return None
            node = node.children[char]
        return node
    
    def get_all_with_prefix(self, prefix: str) -> list:
        """
        Get all words in the trie that start with the given prefix.
        
        Args:
            prefix: The prefix to search for
            
        Returns:
            List of all words starting with the prefix
            
        Time Complexity: O(p + k) where p is prefix length, k is number of results
        """
        results = []
        node = self._find_node(prefix)
        
        if node is None:
            return results
        
        # DFS to collect all words from this node
        self._collect_words(node, prefix, results)
        return results
    
    def _collect_words(self, node: TrieNode, current_word: str, results: list) -> None:
        """
        Recursively collect all words from a given node using DFS.
        
        Args:
            node: Current TrieNode
            current_word: The word built so far
            results: List to append found words to
        """
        if node.is_end:
            results.append(current_word)
        
        for char, child in sorted(node.children.items()):
            self._collect_words(child, current_word + char, results)
    
    def delete(self, word: str) -> bool:
        """
        Delete a word from the trie.
        
        Args:
            word: The word to delete
            
        Returns:
            True if word was deleted, False if word didn't exist
            
        Time Complexity: O(m)
        """
        def _delete_helper(node: TrieNode, word: str, depth: int) -> bool:
            if depth == len(word):
                if not node.is_end:
                    return False  # Word doesn't exist
                node.is_end = False
                return len(node.children) == 0  # Can delete if no children
            
            char = word[depth]
            if char not in node.children:
                return False  # Word doesn't exist
            
            should_delete_child = _delete_helper(node.children[char], word, depth + 1)
            
            if should_delete_child:
                del node.children[char]
                return len(node.children) == 0 and not node.is_end
            
            return False
        
        _delete_helper(self.root, word, 0)
        return True
    
    def count_words_with_prefix(self, prefix: str) -> int:
        """
        Count how many words start with the given prefix.
        
        Args:
            prefix: The prefix to count
            
        Returns:
            Number of words starting with prefix
        """
        node = self._find_node(prefix)
        if node is None:
            return 0
        
        count = [0]
        
        def _count_words(node: TrieNode):
            if node.is_end:
                count[0] += 1
            for child in node.children.values():
                _count_words(child)
        
        _count_words(node)
        return count[0]
    
    def get_all_words(self) -> list:
        """
        Get all words stored in the trie.
        
        Returns:
            List of all words in alphabetical order
        """
        return self.get_all_with_prefix("")

---

## Trie Visualization Helper

In [ ]:
def visualize_trie(trie: Trie) -> None:
    """
    Print a visual representation of the trie structure.
    
    Args:
        trie: The Trie object to visualize
    """
    def _visualize(node: TrieNode, prefix: str, is_last: bool, indent: str):
        # Determine the connector
        connector = "└── " if is_last else "├── "
        
        # Print current node
        if prefix:  # Don't print for root
            end_marker = "*" if node.is_end else ""
            print(f"{indent}{connector}{prefix[-1]}{end_marker}")
        else:
            print("(root)")
        
        # Update indent for children
        if prefix:
            indent += "    " if is_last else "│   "
        
        # Process children
        children = sorted(node.children.items())
        for i, (char, child) in enumerate(children):
            is_last_child = (i == len(children) - 1)
            _visualize(child, prefix + char, is_last_child, indent)
    
    _visualize(trie.root, "", True, "")
    print("\n* = end of word")

---

## Examples

### Example 1: Basic Operations

In [ ]:
# Create a trie and insert words
trie = Trie()
words = ["cat", "car", "card", "dog", "do"]

for word in words:
    trie.insert(word)
    print(f"Inserted: '{word}'")

print("\nTrie structure:")
visualize_trie(trie)

In [ ]:
# Search operations
print("Search operations:")
print(f"  search('cat')  = {trie.search('cat')}   # exact word exists")
print(f"  search('ca')   = {trie.search('ca')}  # prefix only, not a word")
print(f"  search('dogs') = {trie.search('dogs')}  # word doesn't exist")

print("\nPrefix operations:")
print(f"  starts_with('ca')  = {trie.starts_with('ca')}   # prefix exists")
print(f"  starts_with('xyz') = {trie.starts_with('xyz')}  # prefix doesn't exist")

### Example 2: Autocomplete

In [ ]:
# Build autocomplete trie
autocomplete_trie = Trie()
dictionary = [
    "apple", "application", "apply", "apt",
    "banana", "band", "bandana",
    "cat", "car", "card", "care", "careful", "careless",
    "dog", "door", "doom"
]

for word in dictionary:
    autocomplete_trie.insert(word)

print("Dictionary loaded with", len(dictionary), "words")
print("\nTrie structure:")
visualize_trie(autocomplete_trie)

In [ ]:
def autocomplete(trie: Trie, prefix: str, max_results: int = 5) -> list:
    """
    Get autocomplete suggestions for a prefix.
    
    Args:
        trie: The Trie containing words
        prefix: User input to autocomplete
        max_results: Maximum suggestions to return
        
    Returns:
        List of suggested completions
    """
    suggestions = trie.get_all_with_prefix(prefix)
    return suggestions[:max_results]

# Test autocomplete
test_prefixes = ["app", "car", "do", "b", "xyz"]

print("Autocomplete Suggestions:")
print("=" * 40)
for prefix in test_prefixes:
    suggestions = autocomplete(autocomplete_trie, prefix)
    print(f"  '{prefix}' → {suggestions}")

### Example 3: Word Count with Prefix

In [ ]:
print("Word count by prefix:")
print("=" * 40)
prefixes_to_count = ["a", "app", "car", "care", "d", "do", "dog"]

for prefix in prefixes_to_count:
    count = autocomplete_trie.count_words_with_prefix(prefix)
    words = autocomplete_trie.get_all_with_prefix(prefix)
    print(f"  '{prefix}': {count} words → {words}")

### Example 4: Delete Operation

In [ ]:
# Demonstrate delete
delete_trie = Trie()
for word in ["car", "card", "care"]:
    delete_trie.insert(word)

print("Before deletion:")
print(f"  All words: {delete_trie.get_all_words()}")
print(f"  search('car'): {delete_trie.search('car')}")

# Delete 'car' but keep 'card' and 'care'
delete_trie.delete('car')

print("\nAfter deleting 'car':")
print(f"  All words: {delete_trie.get_all_words()}")
print(f"  search('car'): {delete_trie.search('car')}")
print(f"  search('card'): {delete_trie.search('card')}")
print(f"  starts_with('car'): {delete_trie.starts_with('car')}")

---

## Applications of Tries

### 1. Autocomplete Systems
- Search engines (Google, Bing)
- IDE code completion
- Mobile keyboard suggestions

### 2. Spell Checkers
- Fast dictionary lookup
- Suggest corrections by finding similar prefixes

### 3. IP Routing (Longest Prefix Match)
- Network routers use tries to match IP addresses
- Find the most specific route

### 4. Word Games
- Scrabble word validation
- Boggle solver

### 5. DNA Sequence Matching
- Genomics applications
- Pattern matching in biological data

### Application Example: Simple Spell Checker

In [ ]:
class SpellChecker:
    """
    A simple spell checker using a Trie.
    
    Suggests corrections based on prefix matching.
    """
    
    def __init__(self, dictionary: list):
        """Initialize with a list of valid words."""
        self.trie = Trie()
        for word in dictionary:
            self.trie.insert(word.lower())
    
    def check(self, word: str) -> bool:
        """Check if a word is spelled correctly."""
        return self.trie.search(word.lower())
    
    def suggest(self, word: str, max_suggestions: int = 5) -> list:
        """
        Suggest corrections for a misspelled word.
        
        Strategy: Try progressively shorter prefixes until we find matches.
        """
        word = word.lower()
        
        # Try exact prefix first, then shorter ones
        for i in range(len(word), 0, -1):
            prefix = word[:i]
            suggestions = self.trie.get_all_with_prefix(prefix)
            if suggestions:
                # Sort by length difference from original word
                suggestions.sort(key=lambda x: abs(len(x) - len(word)))
                return suggestions[:max_suggestions]
        
        return []

In [ ]:
# Test spell checker
spell_checker = SpellChecker(dictionary)

test_words = ["apple", "aple", "appli", "carz", "bananna", "dog"]

print("Spell Checker Demo:")
print("=" * 50)
for word in test_words:
    is_correct = spell_checker.check(word)
    status = "✓" if is_correct else "✗"
    print(f"  '{word}': {status}", end="")
    if not is_correct:
        suggestions = spell_checker.suggest(word)
        print(f" → suggestions: {suggestions}", end="")
    print()

---

## Space Optimization: Compressed Trie (Radix Tree)

### Problem with Standard Trie
When many nodes have only one child, we waste space:

```
Standard Trie for ["romane", "romanus", "romulus"]:

        (root)
           |
           r
           |
           o
           |
           m        ← 3 nodes just for "rom"
          / \
         a   u
         |   |
         n   l
         |   |
         ...  ...
```

### Compressed Trie (Radix Tree)
Merge chains of single-child nodes:

```
Compressed Trie for same words:

        (root)
           |
         "rom"     ← merged into single edge
          / \
       "an"  "ulus"*
        /\
      "e"* "us"*

* = end of word
```

### Benefits
- **Less memory**: Fewer nodes for sparse tries
- **Same time complexity**: O(m) operations
- **Used in**: Linux kernel routing, many databases

In [ ]:
class CompressedTrieNode:
    """
    Node for compressed trie (radix tree).
    
    Each edge can store a string (not just a single character).
    """
    
    def __init__(self):
        self.children = {}  # first_char -> (edge_string, node)
        self.is_end = False


class CompressedTrie:
    """
    A compressed trie (radix tree) that merges single-child chains.
    
    More space-efficient than standard trie for sparse data.
    """
    
    def __init__(self):
        self.root = CompressedTrieNode()
    
    def insert(self, word: str) -> None:
        """Insert a word into the compressed trie."""
        if not word:
            self.root.is_end = True
            return
        
        node = self.root
        i = 0
        
        while i < len(word):
            char = word[i]
            
            if char not in node.children:
                # Create new edge with remaining string
                new_node = CompressedTrieNode()
                new_node.is_end = True
                node.children[char] = (word[i:], new_node)
                return
            
            edge, child = node.children[char]
            
            # Find common prefix length
            j = 0
            while j < len(edge) and i + j < len(word) and edge[j] == word[i + j]:
                j += 1
            
            if j == len(edge):
                # Entire edge matches, continue to child
                i += j
                node = child
            else:
                # Need to split the edge
                # Create intermediate node
                split_node = CompressedTrieNode()
                
                # Update parent to point to split node
                node.children[char] = (edge[:j], split_node)
                
                # Add original child as child of split node
                split_node.children[edge[j]] = (edge[j:], child)
                
                # Add new word suffix if any remains
                remaining = word[i + j:]
                if remaining:
                    new_node = CompressedTrieNode()
                    new_node.is_end = True
                    split_node.children[remaining[0]] = (remaining, new_node)
                else:
                    split_node.is_end = True
                return
        
        node.is_end = True
    
    def search(self, word: str) -> bool:
        """Search for a word in the compressed trie."""
        node = self.root
        i = 0
        
        while i < len(word):
            char = word[i]
            
            if char not in node.children:
                return False
            
            edge, child = node.children[char]
            
            # Check if edge matches
            if not word[i:].startswith(edge):
                return False
            
            i += len(edge)
            node = child
        
        return node.is_end
    
    def get_all_words(self) -> list:
        """Get all words in the compressed trie."""
        results = []
        
        def collect(node, prefix):
            if node.is_end:
                results.append(prefix)
            for char in sorted(node.children.keys()):
                edge, child = node.children[char]
                collect(child, prefix + edge)
        
        collect(self.root, "")
        return results

In [ ]:
# Compare standard trie vs compressed trie
words = ["romane", "romanus", "romulus", "rubens", "ruber", "rubicon"]

# Standard trie
standard = Trie()
for w in words:
    standard.insert(w)

# Compressed trie
compressed = CompressedTrie()
for w in words:
    compressed.insert(w)

print("Words:", words)
print("\n" + "="*50)
print("Standard Trie:")
visualize_trie(standard)

print("\n" + "="*50)
print("Compressed Trie words:", compressed.get_all_words())
print("\nSearch tests:")
for w in ["romane", "roman", "rubicon", "rub"]:
    print(f"  search('{w}'): {compressed.search(w)}")

---

## Summary

### Key Takeaways

1. **Trie = Prefix Tree**: Optimized for string operations
2. **O(m) operations**: Independent of dictionary size
3. **Natural prefix support**: Autocomplete, spell check
4. **Trade-off**: More memory than hash table, but faster prefix operations

### When to Use a Trie

| Use Trie | Use Hash Table |
|----------|----------------|
| Prefix search needed | Exact match only |
| Autocomplete features | Simple key-value lookup |
| Sorted output needed | Order doesn't matter |
| Many shared prefixes | Random string keys |

### Related Data Structures

- **Radix Tree**: Compressed trie (covered above)
- **Ternary Search Tree**: Space-efficient alternative
- **Suffix Tree**: For substring matching
- **Aho-Corasick Automaton**: Multi-pattern matching (extends trie with failure links)

---

## Exercises

### Exercise 1: Gene Name Autocomplete (*)

In genomics databases, gene names often share common prefixes (e.g., BRCA1, BRCA2, BRAF, BRD4 all start with "BR"). A trie is a natural fit for prefix-based lookup.

**Task:** Build a trie from the gene name list below and implement `gene_autocomplete(trie, prefix)` that returns all gene names matching the given prefix in alphabetical order.

Test with prefixes: `"BR"`, `"TP"`, `"MYC"`, `"Z"` (no matches).

In [ ]:
GENE_NAMES = [
    "BRCA1", "BRCA2", "BRAF", "BRD4", "BRDT",
    "TP53", "TP63", "TP73",
    "MYC", "MYCN", "MYCL",
    "EGFR", "KRAS", "NRAS", "HRAS",
    "PIK3CA", "PIK3CB", "PIK3CD",
    "ATM", "ATR",
]


def gene_autocomplete(trie: Trie, prefix: str) -> list[str]:
    """
    Return all gene names in the trie that start with prefix.

    Args:
        trie: A Trie loaded with gene names
        prefix: The prefix to search for (case-sensitive)

    Returns:
        Alphabetically sorted list of matching gene names
    """
    # YOUR CODE HERE
    pass


# gene_trie = Trie()
# for gene in GENE_NAMES:
#     gene_trie.insert(gene)
#
# for prefix in ["BR", "TP", "MYC", "Z"]:
#     matches = gene_autocomplete(gene_trie, prefix)
#     print(f"Prefix '{prefix}': {matches}")

### Exercise 2: Counting Unique K-mers with a Trie (**)

K-mers (substrings of length k) are fundamental in sequence analysis — used for genome assembly, read mapping, and repeat detection.

**Task:** Implement `count_unique_kmers(sequence, k)` using a trie: insert every k-mer, then count leaf-reachable end-of-word nodes. Verify your result against `len(set(...))` for the same sequence.

Note how the trie stores prefix structure, while the set approach does not.

In [ ]:
DNA = "ATCGATCGATCGAATTCCGATCGATCGATCG"


def count_unique_kmers_trie(sequence: str, k: int) -> tuple[int, list[str]]:
    """
    Count unique k-mers in a DNA sequence using a trie.

    Args:
        sequence: DNA string (alphabet A/T/C/G)
        k: k-mer length

    Returns:
        Tuple of (unique_kmer_count, sorted list of unique k-mers)
    """
    # YOUR CODE HERE
    pass


def count_unique_kmers_set(sequence: str, k: int) -> int:
    """Count unique k-mers using a set (reference implementation)."""
    # YOUR CODE HERE
    pass


# for k in [2, 3, 4, 5]:
#     trie_count, _ = count_unique_kmers_trie(DNA, k)
#     set_count = count_unique_kmers_set(DNA, k)
#     match = "OK" if trie_count == set_count else "MISMATCH"
#     print(f"k={k}: trie={trie_count}, set={set_count} [{match}]")

---[< Previous: DFA-Based Pattern Matching](../07_String_Algorithms/04_dfa_matching.ipynb) | [Tier 4: Algorithms & Data Structures Overview](../README.md) | [Next: Aho-Corasick Algorithm >](02_aho_corasick.ipynb)---